# 05 Maps: where the monotown types are

**Goal:** map the six monotown types from notebook 04 and show Slantsy's place among them.

**Data:** `data/processed/clusters.csv` (232 towns with their type). The files contain no geometry, so town coordinates are found by **geocoding**: each town name with its region is sent to Nominatim, the search service of OpenStreetMap (© OpenStreetMap contributors). Results are cached in `data/processed/coords.csv`, so Nominatim is queried only once per town (it allows one request per second). Common names (Lesnoy, Gorny, Yasny...) can match the wrong place, so every result is checked against the expected region and corrected by hand where needed.

Each point marks the town itself, not the boundaries of its municipality.

In [2]:
import os
import pandas as pd
import folium

from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter

In [3]:
# geocode each town once with Nominatim; results are cached in coords.csv
COORDS = "../data/processed/coords.csv"
clusters = pd.read_csv("../data/processed/clusters.csv", dtype={"oktmo": str})

if os.path.exists(COORDS):
    coords = pd.read_csv(COORDS)
else:
    coords = pd.DataFrame(columns=["number", "query", "lat", "lon", "found_name"])

todo = clusters[~clusters["number"].isin(coords["number"])]
print("Towns to geocode:", len(todo))

geolocator = Nominatim(user_agent="russian-monotowns-typology (student project)")
geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1.1)

rows = []
for _, t in todo.iterrows():
    query = f"{t['name_en']}, {t['region_en']}, Russia"
    loc = geocode(query, language="en", country_codes="ru", featuretype="settlement")
    rows.append({"number": t["number"], "query": query,
                 "lat": loc.latitude if loc else None,
                 "lon": loc.longitude if loc else None,
                 "found_name": loc.address if loc else None})

if rows:
    new = pd.DataFrame(rows)
    coords = new if coords.empty else pd.concat([coords, new], ignore_index=True)
    coords.to_csv(COORDS, index=False)

print("Not found:", coords["lat"].isna().sum())

Towns to geocode: 232
Not found: 8


In [5]:
# second try for towns not found under their English names: Russian names
RU_QUERIES = {
    20: "Демьяново, Кировская область",
    76: "Верх-Усугли, Забайкальский край",
    100: "Белая Холуница, Кировская область",
    134: "Ярославский, Приморский край",
    161: "Камские Поляны, Республика Татарстан",
    184: "Увельский, Челябинская область",
    212: "Медногорский, Карачаево-Черкесская Республика",
    225: "Княгинино, Нижегородская область",
}
# last resort: coordinates looked up by hand on openstreetmap.org, as number: (lat, lon)
MANUAL = {}

changed = False
for num, query in RU_QUERIES.items():
    row = coords["number"] == num
    if coords.loc[row, "lat"].isna().all():
        loc = geocode(query, language="en", country_codes="ru")
        if loc:
            coords.loc[row, ["query", "lat", "lon", "found_name"]] = \
                [query, loc.latitude, loc.longitude, loc.address]
            changed = True

for num, (lat, lon) in MANUAL.items():
    region = clusters.loc[clusters["number"] == num, "region_en"].iloc[0]
    coords.loc[coords["number"] == num, ["lat", "lon", "found_name"]] = \
        [lat, lon, f"set by hand, {region}"]
    changed = True

if changed:
    coords.to_csv(COORDS, index=False)
print("Still not found:", coords.loc[coords["lat"].isna(), "number"].tolist())

Still not found: []


In [8]:
# interactive map of the six types; click a point for details, star = Slantsy
towns = clusters.merge(coords[["number", "lat", "lon"]], on="number").dropna(subset=["lat"])

COLORS = {
    "Depopulating towns": "#d62728",
    "Working towns losing people": "#ff7f0e",
    "Slow decline, thin job base": "#9467bd",
    "Stable industrial centres": "#1f77b4",
    "High-employment strongholds": "#2ca02c",
    "Job-boom towns": "#17becf",
}

m = folium.Map(
    location=[60, 90], zoom_start=3,
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/Canvas/"
          "World_Light_Gray_Base/MapServer/tile/{z}/{y}/{x}",
    attr="Tiles © Esri, HERE, Garmin, © OpenStreetMap contributors",
)
for _, t in towns.iterrows():
    folium.CircleMarker(
        location=[t["lat"], t["lon"]], radius=5,
        color=COLORS[t["type"]], fill=True, fill_opacity=0.85, weight=1,
        popup=f"<b>{t['name_en']}</b><br>{t['region_en']}<br>{t['type']}",
    ).add_to(m)

sl = towns[towns["number"] == 109].iloc[0]
folium.Marker([sl["lat"], sl["lon"]], tooltip="Slantsy",
              icon=folium.Icon(color="black", icon="star")).add_to(m)

# legend in the top-right corner (Arctic Ocean, no towns there)
legend = "".join(f'<div><span style="color:{c}; font-size:18px">&#9679;</span> {t}</div>'
                 for t, c in COLORS.items())
m.get_root().html.add_child(folium.Element(
    f'<div style="position:fixed; top:20px; right:20px; z-index:9999; background:white; '
    f'padding:8px 12px; border:1px solid #ccc; font-size:13px">'
    f'<b>Russian monotown types</b><br>232 towns, k-means clustering<br>'
    f'<span style="font-size:11px">click a point for details, star = Slantsy</span>'
    f'<hr style="margin:6px 0">{legend}</div>'))

# save as a standalone web page for GitHub Pages
os.makedirs("../docs", exist_ok=True)
m.save("../docs/monotown_types_map.html")
m

In [4]:
#rough check: does the found address mention the expected region?
GENERIC = {"oblast", "republic", "krai", "of", "the", "autonomous", "okrug"}

def region_ok(region, address):
    if not isinstance(address, str):
        return False
    keys = [w.strip("()").lower() for w in region.split() if w.lower() not in GENERIC]
    return any(k in address.lower() for k in keys)

check = clusters.merge(coords, on="number")
check["region_ok"] = [region_ok(r, a) for r, a in zip(check["region_en"], check["found_name"])]

print("Missing or suspicious:", (~check["region_ok"]).sum())
check.loc[~check["region_ok"], ["number", "name_en", "region_en", "found_name"]]

Missing or suspicious: 8


,number,name_en,region_en,found_name
14,20,Demyanovo,Kirov Oblast,NaN
63,76,Verkh-Usugli,Zabaykalsky Krai,NaN
87,100,Belaya Kholunitsa,Kirov Oblast,NaN
119,134,Yaroslavsky,Primorsky Krai,NaN
137,161,Kamskiye Polyany,Republic of Tatarstan,NaN
157,184,Uvelsky,Chelyabinsk Oblast,NaN
183,212,Mednogorsky,Karachay-Cherkess Republic,NaN
196,225,Knyaginino,Nizhny Novgorod Oblast,NaN
